# 05 — Target Definition & Temporal Validation

**Wind Turbine Predictive Maintenance & Failure Intelligence System**

## Why this is the most important notebook
A predictive-maintenance model is only as trustworthy as its target and its
validation. This notebook makes two decisions that determine whether the whole
project is honest or fake:

1. **Target:** a binary label — is a row inside (or in the run-up to) a real fault
   window? Built from `event_start_id`/`event_end_id` (NB01's verified mapping),
   **never** from `status_type_id` (NB02: entangled with the fault → leakage).

2. **Validation:** with only 12 events across 5 turbines, a random split leaks
   massively (rows from one event land in both train and test). We use
   **leave-turbines-out**: entire turbines are held out for testing, so the model
   must generalise to a turbine it has never seen — the realistic deployment case.

## Design decisions made here (with reasoning)
- Lead-time window: label the N hours *before* `event_start` as positive too,
  since degradation precedes the official window.
- Normal runs contribute only negatives.
- Split is by `asset_id`, not by row — justified and demonstrated.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

BASE = Path("..") / "data" / "raw" / "Wind Farm A"
FEATURES_DIR = Path("..") / "data" / "processed" / "features"

events = pd.read_csv(BASE / "event_info.csv", sep=";")
feature_list = pd.read_csv(FEATURES_DIR / "feature_list.csv")["feature"].tolist()

# The feature matrix is large — load it and check structure
fm = pd.read_csv(FEATURES_DIR / "feature_matrix.csv")
print("Feature matrix shape:", fm.shape)
print("Key columns:", [c for c in fm.columns if c in ["id","asset_id","event_id"]])
print("Number of features:", len(feature_list))
print("\nRows per event_id:")
print(fm["event_id"].value_counts().sort_index())

Feature matrix shape: (1195779, 187)
Key columns: ['id', 'asset_id', 'event_id']
Number of features: 184

Rows per event_id:
event_id
0     54942
3     55443
10    53548
13    53966
14    54153
17    55046
22    52992
24    54959
25    54668
26    53658
38    54791
40    56114
42    53842
45    53695
51    54392
68    54314
69    54769
71    54700
72    54038
73    53998
84    53728
92    54023
Name: count, dtype: int64


## 1. Build the binary target

The label answers: *is this row within the pre-fault window of a real fault event?*

Construction (per the NB01-verified mapping):
- For **anomaly** events: rows where `id` is inside `[event_start_id − LEAD,
  event_end_id]` are positive (1). The `LEAD` extends the window *backward* to
  capture early degradation before the official start.
- For **normal** events: all rows are negative (0) — no fault occurred.
- `status_type_id` is never used to build the label (leakage).

We start with **LEAD = 144 steps (24 h)** — labelling the day before the official
window as also "pre-fault", a reasonable predictive horizon. We check the
resulting class balance.

In [2]:
LEAD = 144  # 24h lead-time before the official window start

# Map event_id -> (label, asset, start_id, end_id)
ev = events.set_index("event_id")

def build_target(fm, lead):
    fm = fm.copy()
    y = np.zeros(len(fm), dtype=int)

    for event_id in fm["event_id"].unique():
        row = ev.loc[event_id]
        mask_event = fm["event_id"] == event_id
        if row["event_label"] == "anomaly":
            start = row["event_start_id"] - lead
            end = row["event_end_id"]
            pos = mask_event & (fm["id"] >= start) & (fm["id"] <= end)
            y[pos.values] = 1
        # normal events: leave as 0
    return y

fm["target"] = build_target(fm, LEAD)

print("Target distribution:")
print(fm["target"].value_counts())
print(f"\nPositive class: {fm['target'].mean():.2%} of all rows")

print("\nPositive rows per event (anomaly events should have positives, normal = 0):")
chk = fm.groupby("event_id").agg(
    label=("target", lambda s: "has_pos" if s.sum() > 0 else "all_neg"),
    n_pos=("target", "sum"),
).join(ev[["event_label", "event_description"]])
print(chk.to_string())

Target distribution:
target
0    1175811
1      19968
Name: count, dtype: int64

Positive class: 1.67% of all rows

Positive rows per event (anomaly events should have positives, normal = 0):
            label  n_pos event_label          event_description
event_id                                                       
0         has_pos   2156     anomaly  Generator bearing failure
3         all_neg      0      normal                        NaN
10        has_pos   1125     anomaly            Gearbox failure
13        all_neg      0      normal                        NaN
14        all_neg      0      normal                        NaN
17        all_neg      0      normal                        NaN
22        has_pos   1149     anomaly            Hydraulic group
24        all_neg      0      normal                        NaN
25        all_neg      0      normal                        NaN
26        has_pos   1153     anomaly            Hydraulic group
38        all_neg      0      normal    

## 2. Validation design — why random splitting fails here

The instinct is a random train/test split. Here it is **invalid and dangerous**:

- Rows are 10 min apart and highly autocorrelated. A random split puts row *t* in
  train and *t+1* in test — the model effectively sees the answer. Scores look
  near-perfect and mean nothing.
- With only **12 events**, the model can memorise each specific event rather than
  learn generalisable degradation patterns.

The realistic deployment question is: *given turbines we've learned from, will we
catch a fault on a **different** turbine?* So we validate with **leave-turbines-out**:
hold out all data from one or more turbines for testing; train on the rest. The
model must generalise to a turbine it has never seen — exactly the deployment case.

We have 5 turbines (assets 0, 10, 11, 13, 21). We use **GroupKFold on `asset_id`**,
so no turbine appears in both train and test in any fold.

In [3]:
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

# Small, fast demo model to expose the split difference (not the real model — that's NB06)
X = fm[feature_list].values
y = fm["target"].values
groups = fm["asset_id"].values

def quick_ap(train_idx, test_idx):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X[train_idx])
    Xte = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=200, class_weight="balanced")
    clf.fit(Xtr, y[train_idx])
    proba = clf.predict_proba(Xte)[:, 1]
    return average_precision_score(y[test_idx], proba)

# Use a subsample for speed in this demonstration
rng = np.random.default_rng(42)
idx = rng.choice(len(fm), size=150000, replace=False)
Xs, ys, gs = X[idx], y[idx], groups[idx]
X, y, groups = Xs, ys, gs  # rebind for the demo functions

# (a) RANDOM split (stratified) — the WRONG way
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_aps = [quick_ap(tr, te) for tr, te in skf.split(X, y)]

# (b) GROUPED split (leave-turbines-out) — the RIGHT way
gkf = GroupKFold(n_splits=5)
grouped_aps = [quick_ap(tr, te) for tr, te in gkf.split(X, y, groups)]

print("Baseline positive rate:", round(y.mean(), 4))
print(f"\nRandom split  PR-AUC: {np.mean(random_aps):.3f}  (per fold: {[round(a,3) for a in random_aps]})")
print(f"Grouped split PR-AUC: {np.mean(grouped_aps):.3f}  (per fold: {[round(a,3) for a in grouped_aps]})")
print("\nThe random split's inflated score is the leakage. The grouped score is honest.")

/opt/anaconda3/envs/windturbine/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/windturbine/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mode

Baseline positive rate: 0.0164

Random split  PR-AUC: 0.059  (per fold: [0.058, 0.06, 0.057, 0.063, 0.055])
Grouped split PR-AUC: 0.033  (per fold: [0.02, 0.031, 0.036, 0.068, 0.012])

The random split's inflated score is the leakage. The grouped score is honest.


### Validation findings

- **Leave-turbines-out scores lower than random** (grouped PR-AUC 0.033 vs random
  0.059, both on a quick logistic baseline). The gap confirms the random split is
  optimistic — information leaks across autocorrelated rows of the same turbine —
  so **grouped validation is the correct, honest choice**. All modelling in NB06
  uses GroupKFold on `asset_id`.
- **The absolute scores are low** (grouped ≈ 2× the 0.016 baseline). A linear model
  barely separates pre-fault from normal. This is expected and informative:
  - The signal is fault-specific and buried (NB02), not linearly separable.
  - One binary model must cover 4 different fault mechanisms living in different
    sensors → hard by construction.
  - Motivates **non-linear models (XGBoost) in NB06** and **anomaly detection as a
    co-lead (NB07)**, rather than expecting a clean linear classifier.
- **Honest framing:** this is a genuinely hard real-world problem with 12 events.
  We report real leave-turbines-out performance, not inflated random-split numbers.